## 9. Final Ensemble Summary

✅ **UNIFIED ENSEMBLE SYSTEM COMPLETE**

### 🎯 Key Results

**Individual Models:**
- LSTM, RNN (deep learning) and Random Forest, Gaussian (traditional ML) trained on identical data
- Each model produces predictions with the same shape and scale

**Ensemble Methods:**
1. **Simple Average**: Equal weight (0.25 each)
2. **Weighted Average**: Performance-based weights (R² optimized)

### 📊 Benefits of Ensemble Approach

✓ **Robustness**: Combines strengths of all 4 different architectures  
✓ **Reduced Overfitting**: Averaging reduces individual model bias  
✓ **Generalization**: Handles diverse patterns better than any single model  
✓ **Interpretability**: Can analyze which models contribute most  

### 🚀 Next Steps

- Deploy the weighted ensemble for production forecasting
- Monitor individual model performance and adapt weights dynamically
- Collect new data and retrain models periodically
- Experiment with additional models or ensemble techniques

**The ensemble is ready for deployment!**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# Calculate residuals for all models and ensembles
residuals_dict = {
    'LSTM': y_test_flat - lstm_pred_scaled,
    'RNN': y_test_flat - rnn_pred_scaled,
    'Random Forest': y_test_flat - rf_pred_scaled,
    'Gaussian': y_test_flat - gp_pred_scaled,
    'Simple Ensemble': y_test_flat - ensemble_simple_pred,
    'Weighted Ensemble': y_test_flat - ensemble_weighted_pred
}

# Plot 1: Residuals Distribution
ax1 = axes[0, 0]
for label, residuals in residuals_dict.items():
    ax1.hist(residuals, bins=20, alpha=0.6, label=label, edgecolor='black')
ax1.set_xlabel('Residuals', fontweight='bold')
ax1.set_ylabel('Frequency', fontweight='bold')
ax1.set_title('Residuals Distribution', fontweight='bold', fontsize=12)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3, axis='y')

# Plot 2: Residuals Time Series
ax2 = axes[0, 1]
ax2.plot(residuals_dict['LSTM'], alpha=0.6, linewidth=1, label='LSTM')
ax2.plot(residuals_dict['RNN'], alpha=0.6, linewidth=1, label='RNN')
ax2.plot(residuals_dict['Random Forest'], alpha=0.6, linewidth=1, label='RF')
ax2.plot(residuals_dict['Gaussian'], alpha=0.6, linewidth=1, label='GP')
ax2.plot(residuals_dict['Weighted Ensemble'], linewidth=2, label='Weighted Ensemble', color='#F18F01')
ax2.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Sample', fontweight='bold')
ax2.set_ylabel('Residual', fontweight='bold')
ax2.set_title('Residuals Over Time', fontweight='bold', fontsize=12)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# Plot 3: Error Magnitude Comparison
ax3 = axes[1, 0]
abs_errors = {label: np.abs(residuals) for label, residuals in residuals_dict.items()}
error_stats = pd.DataFrame({
    'Mean': {k: np.mean(v) for k, v in abs_errors.items()},
    'Std': {k: np.std(v) for k, v in abs_errors.items()},
    'Max': {k: np.max(v) for k, v in abs_errors.items()}
})
error_stats['Mean'].sort_values().plot(kind='barh', ax=ax3, color='#2E86AB', edgecolor='black', linewidth=1.5)
ax3.set_xlabel('Mean Absolute Error', fontweight='bold')
ax3.set_title('Mean Absolute Error by Model', fontweight='bold', fontsize=12)
ax3.grid(True, alpha=0.3, axis='x')

# Plot 4: Error Box Plot
ax4 = axes[1, 1]
residuals_for_box = [residuals_dict[k] for k in sorted(residuals_dict.keys())]
bp = ax4.boxplot(residuals_for_box, labels=sorted(residuals_dict.keys()), patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#2E86AB')
    patch.set_alpha(0.7)
ax4.set_ylabel('Residuals', fontweight='bold')
ax4.set_title('Residuals Box Plot', fontweight='bold', fontsize=12)
ax4.axhline(y=0, color='red', linestyle='--', linewidth=2)
ax4.grid(True, alpha=0.3, axis='y')
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

print("✓ Residuals analysis completed")

## 8. Residuals Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: R² Comparison
ax1 = axes[0, 0]
r2_scores = comparison_df['R²'].sort_values(ascending=True)
colors = ['#2E86AB' if 'Ensemble' not in x else '#F18F01' for x in r2_scores.index]
bars1 = ax1.barh(range(len(r2_scores)), r2_scores.values, color=colors, edgecolor='black', linewidth=1.5)
ax1.set_yticks(range(len(r2_scores)))
ax1.set_yticklabels(r2_scores.index)
ax1.set_xlabel('R² Score', fontweight='bold', fontsize=11)
ax1.set_title('R² Score Comparison (Higher is Better)', fontweight='bold', fontsize=12)
ax1.set_xlim(left=0)
ax1.grid(True, alpha=0.3, axis='x')
for i, v in enumerate(r2_scores.values):
    ax1.text(v + 0.01, i, f'{v:.4f}', va='center', fontweight='bold')

# Plot 2: RMSE Comparison
ax2 = axes[0, 1]
rmse_scores = comparison_df['RMSE'].sort_values(ascending=False)
colors2 = ['#2E86AB' if 'Ensemble' not in x else '#F18F01' for x in rmse_scores.index]
bars2 = ax2.barh(range(len(rmse_scores)), rmse_scores.values, color=colors2, edgecolor='black', linewidth=1.5)
ax2.set_yticks(range(len(rmse_scores)))
ax2.set_yticklabels(rmse_scores.index)
ax2.set_xlabel('RMSE', fontweight='bold', fontsize=11)
ax2.set_title('RMSE Comparison (Lower is Better)', fontweight='bold', fontsize=12)
ax2.set_xlim(left=0)
ax2.grid(True, alpha=0.3, axis='x')
for i, v in enumerate(rmse_scores.values):
    ax2.text(v + 0.005, i, f'{v:.4f}', va='center', fontweight='bold')

# Plot 3: All Predictions vs Actual (first 100 samples)
ax3 = axes[1, 0]
x_range = np.arange(100)
ax3.plot(x_range, y_test_flat[:100], 'o-', label='Actual', linewidth=2.5, markersize=4, color='black', zorder=5)
ax3.plot(x_range, lstm_pred_scaled[:100], '--', label='LSTM', linewidth=1.5, alpha=0.7)
ax3.plot(x_range, rnn_pred_scaled[:100], '--', label='RNN', linewidth=1.5, alpha=0.7)
ax3.plot(x_range, rf_pred_scaled[:100], '--', label='RF', linewidth=1.5, alpha=0.7)
ax3.plot(x_range, gp_pred_scaled[:100], '--', label='GP', linewidth=1.5, alpha=0.7)
ax3.plot(x_range, ensemble_simple_pred[:100], '-', label='Simple Ensemble', linewidth=2.5, color='#F18F01')
ax3.plot(x_range, ensemble_weighted_pred[:100], '-', label='Weighted Ensemble', linewidth=2.5, color='#A23B72')
ax3.set_xlabel('Sample', fontweight='bold')
ax3.set_ylabel('Scaled Demand', fontweight='bold')
ax3.set_title('Predictions Comparison (First 100 Samples)', fontweight='bold', fontsize=12)
ax3.legend(loc='best', fontsize=9)
ax3.grid(True, alpha=0.3)

# Plot 4: Ensemble Weights
ax4 = axes[1, 1]
weights = ensemble_weighted.weights
model_names = [k.replace('_', ' ').title() for k in weights.keys()]
weight_values = list(weights.values())
colors4 = ['#2E86AB', '#A23B72', '#F18F01', '#06A77D']
bars4 = ax4.bar(model_names, weight_values, color=colors4, edgecolor='black', linewidth=1.5)
ax4.set_ylabel('Weight', fontweight='bold', fontsize=11)
ax4.set_title('Weighted Ensemble: Model Contributions', fontweight='bold', fontsize=12)
ax4.set_ylim(top=max(weight_values) * 1.15)
ax4.grid(True, alpha=0.3, axis='y')
for i, (bar, weight) in enumerate(zip(bars4, weight_values)):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{weight:.1%}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

print("\n✓ Visualizations created")

## 7. Visualizations

In [ ]:
print("\n" + "="*60)
print("📊 COMPREHENSIVE COMPARISON: All Models + Ensembles")
print("="*60)

# Create comprehensive comparison
comparison_data = {
    'LSTM': metrics['LSTM'],
    'RNN': metrics['RNN'],
    'Random Forest': metrics['Random Forest'],
    'Gaussian': metrics['Gaussian'],
    'Simple Ensemble': {
        'MSE': simple_mse,
        'MAE': simple_mae,
        'RMSE': simple_rmse,
        'R²': simple_r2,
        'MAPE': simple_mape
    },
    'Weighted Ensemble': {
        'MSE': weighted_mse,
        'MAE': weighted_mae,
        'RMSE': weighted_rmse,
        'R²': weighted_r2,
        'MAPE': weighted_mape
    }
}

comparison_df = pd.DataFrame(comparison_data).T
print("\n" + comparison_df.to_string())

# Rank by R² score
print("\n\n🏆 RANKING BY R² SCORE (Higher is Better):")
ranking = comparison_df['R²'].sort_values(ascending=False)
for i, (model, r2) in enumerate(ranking.items(), 1):
    print(f"  {i}. {model}: {r2:.4f}")

## 6. Compare All Methods

In [ ]:
print("\n" + "="*60)
print("🎯 ENSEMBLE METHOD 2: PERFORMANCE-WEIGHTED AVERAGING")
print("="*60)

# Create ensemble with weighted averaging
ensemble_weighted = EnsembleForecaster(method='weighted_average')

# Adapt weights based on individual model R² scores
ensemble_weighted.predict(predictions_dict, y_true=y_test_flat)
ensemble_weighted_pred = ensemble_weighted.predict(predictions_dict)

# Calculate metrics
weighted_mse = mean_squared_error(y_test_flat, ensemble_weighted_pred)
weighted_mae = mean_absolute_error(y_test_flat, ensemble_weighted_pred)
weighted_rmse = np.sqrt(weighted_mse)
weighted_r2 = r2_score(y_test_flat, ensemble_weighted_pred)
weighted_mape = mean_absolute_percentage_error(y_test_flat, ensemble_weighted_pred) * 100

print(f"\n✨ Weighted Average Ensemble Performance:")
print(f"  RMSE: {weighted_rmse:.4f}")
print(f"  MAE:  {weighted_mae:.4f}")
print(f"  R²:   {weighted_r2:.4f}")
print(f"  MAPE: {weighted_mape:.2f}%")

print(f"\n  Adaptive Weights (based on R² scores):")
for model_name, weight in ensemble_weighted.weights.items():
    print(f"    {model_name.replace('_', ' ').title()}: {weight:.4f}")

# Compare to simple average
improvement_vs_simple = ((weighted_r2 - simple_r2) / abs(simple_r2)) * 100
print(f"\n  Improvement vs Simple Average: {improvement_vs_simple:+.2f}%")

## 5. Ensemble: Performance-Based Weighted Averaging

In [ ]:
print("\n" + "="*60)
print("🔗 ENSEMBLE METHOD 1: SIMPLE AVERAGING")
print("="*60)

# Create ensemble with simple averaging
ensemble_simple = EnsembleForecaster(method='simple_average')

predictions_dict = {
    'lstm': lstm_pred_scaled,
    'rnn': rnn_pred_scaled,
    'random_forest': rf_pred_scaled,
    'gaussian': gp_pred_scaled
}

ensemble_simple_pred = ensemble_simple.predict(predictions_dict)

# Calculate metrics
simple_mse = mean_squared_error(y_test_flat, ensemble_simple_pred)
simple_mae = mean_absolute_error(y_test_flat, ensemble_simple_pred)
simple_rmse = np.sqrt(simple_mse)
simple_r2 = r2_score(y_test_flat, ensemble_simple_pred)
simple_mape = mean_absolute_percentage_error(y_test_flat, ensemble_simple_pred) * 100

print(f"\n✨ Simple Average Ensemble Performance:")
print(f"  RMSE: {simple_rmse:.4f}")
print(f"  MAE:  {simple_mae:.4f}")
print(f"  R²:   {simple_r2:.4f}")
print(f"  MAPE: {simple_mape:.2f}%")
print(f"\n  Ensemble weights: Equal (0.25 each)")

# Compare to best individual model
best_r2 = metrics_df['R²'].max()
improvement = ((simple_r2 - best_r2) / abs(best_r2)) * 100
print(f"\n  Improvement vs Best Individual Model: {improvement:+.2f}%")

## 4. Ensemble: Simple Averaging

In [ ]:
print("\n" + "="*60)
print("📈 INDIVIDUAL MODEL PERFORMANCE")
print("="*60)

# Calculate metrics for each model
metrics = {}
models_names = ['LSTM', 'RNN', 'Random Forest', 'Gaussian']
predictions_list = [
    (lstm_pred_scaled, lstm_predictions),
    (rnn_pred_scaled, rnn_predictions),
    (rf_pred_scaled, rf_predictions),
    (gp_pred_scaled, gp_predictions)
]

for name, (pred_scaled, pred_orig) in zip(models_names, predictions_list):
    mse = mean_squared_error(y_test_flat, pred_scaled)
    mae = mean_absolute_error(y_test_flat, pred_scaled)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_flat, pred_scaled)
    mape = mean_absolute_percentage_error(y_test_flat, pred_scaled) * 100
    
    metrics[name] = {'MSE': mse, 'MAE': mae, 'RMSE': rmse, 'R²': r2, 'MAPE': mape}
    
    print(f"\n{name}:")
    print(f"  RMSE: {rmse:.4f}  |  MAE: {mae:.4f}  |  R²: {r2:.4f}  |  MAPE: {mape:.2f}%")

# Create comparison DataFrame
metrics_df = pd.DataFrame(metrics).T
print("\n" + metrics_df.to_string())

## 3. Individual Model Performance

In [ ]:
print("\n" + "="*60)
print("🚀 TRAINING ALL 4 MODELS WITH UNIFIED DATA")
print("="*60)

# Model 1: LSTM
print("\n1️⃣  Training LSTM...")
lstm = lstm_model.LSTMModel(seq_length=data['seq_length'], n_features=data['n_features'], 
                             epochs=30, verbose=0)
lstm.build_model()
lstm.model.fit(X_train_seq, y_train_seq, epochs=30, batch_size=16, 
               validation_split=0.1, verbose=0)
lstm_pred_scaled = lstm.model.predict(X_test_seq, verbose=0).flatten()
lstm_predictions = preprocessing.inverse_scale_predictions(lstm_pred_scaled, scaler_y)
print(f"   ✓ LSTM Output shape: {lstm_predictions.shape}")

# Model 2: RNN
print("2️⃣  Training RNN...")
rnn = rnn_model.RNNModel(seq_length=data['seq_length'], n_features=data['n_features'],
                          epochs=30, verbose=0)
rnn.build_model()
rnn.model.fit(X_train_seq, y_train_seq, epochs=30, batch_size=16,
              validation_split=0.1, verbose=0)
rnn_pred_scaled = rnn.model.predict(X_test_seq, verbose=0).flatten()
rnn_predictions = preprocessing.inverse_scale_predictions(rnn_pred_scaled, scaler_y)
print(f"   ✓ RNN Output shape: {rnn_predictions.shape}")

# Model 3: Random Forest
print("3️⃣  Training Random Forest...")
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train_flat, y_train_flat)
rf_pred_scaled = rf.predict(X_test_flat)
rf_predictions = preprocessing.inverse_scale_predictions(rf_pred_scaled, scaler_y)
print(f"   ✓ Random Forest Output shape: {rf_predictions.shape}")

# Model 4: Gaussian Process
print("4️⃣  Training Gaussian Process...")
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
kernel = C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2))
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True, random_state=42)
gp.fit(X_train_flat, y_train_flat)
gp_pred_scaled, _ = gp.predict(X_test_flat, return_std=True)
gp_predictions = preprocessing.inverse_scale_predictions(gp_pred_scaled, scaler_y)
print(f"   ✓ Gaussian Process Output shape: {gp_predictions.shape}")

print("\n✅ All 4 models trained successfully with UNIFIED data!")
print("✅ All predictions have IDENTICAL shape!")

## 2. Train All 4 Models in Parallel

In [ ]:
print("📊 Loading unified data for all models...\n")

# Load data (SAME for all models - this is crucial for ensemble)
X, y = preprocessing.load_synthetic_data(n_samples=500, n_features=10)

# UNIFIED preprocessing
data = preprocessing.preprocess_data(X, y, test_size=0.2, seq_length=10)

# Extract all required formats
X_train_seq = data['X_train_seq']
X_test_seq = data['X_test_seq']
y_train_seq = data['y_train_seq']
y_test_seq = data['y_test_seq']

X_train_flat = data['X_train_flat']
X_test_flat = data['X_test_flat']
y_train_flat = data['y_train_flat']
y_test_flat = data['y_test_flat']

scaler_y = data['scaler_y']

print(f"✓ Unified Data Loaded:")
print(f"  Sequences (for LSTM/RNN): {X_train_seq.shape} → {X_test_seq.shape}")
print(f"  Flat (for RF/GP): {X_train_flat.shape} → {X_test_flat.shape}")
print(f"  Target (all models): {y_train_flat.shape} → {y_test_flat.shape}")
print(f"\n✓ All models will use IDENTICAL input and target data")

## 1. Load and Preprocess Data (Unified)

In [ ]:
import sys
sys.path.insert(0, '../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam

from src import preprocessing
from src.ensemble import EnsembleForecaster
from src.models import lstm_model, rnn_model, random_forest_model, gaussian_model

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print("✓ Libraries and modules imported successfully")
print("✓ Ensemble system initialized")

# Demand Forecasting System - Unified Ensemble

## Overview
This notebook combines predictions from **all 4 models** (LSTM, RNN, Random Forest, Gaussian Process) into a single unified ensemble forecast.

**Key Concept:**
- Models are NOT independent → They work **together** as an ensemble
- Each model contributes to the final prediction
- Ensemble methods increase robustness and reduce individual model bias

**Ensemble Methods:**
1. **Simple Averaging**: Equal weight for all models
2. **Weighted Averaging**: Weights based on individual model performance
3. **Performance-Based Adaptation**: Dynamically adjust weights using validation data